# Module 2 — Les fondamentaux du machine learning

**CONSEIL :** Sauvegardez une copie de ce notebook dans votre Drive avant de commencer !

**Objectifs d'apprentissage**
- Comprendre ce qu'est une **feature** (et pourquoi il en faut)
- Savoir **nettoyer** un dataset sans pleurer
- Faire un **train/test split** sans tricher
- Diagnostiquer un **overfitting** à l'œil nu
- Saisir le **compromis biais-variance** (le seul vrai mystère du ML)

## Le running example : Law School

Un dataset classique en sociologie quantitative du droit et en fairness ML : la *LSAC National Longitudinal Bar Passage Study* (Wightman, 1998). On suit environ 18 700 étudiant·es de droit aux États-Unis, depuis leur entrée en faculté jusqu'à leur passage du barreau (l'examen national pour exercer comme avocat·e).

Le parcours :

```
LSAT + UGPA → 1ère année → ... → Law School GPA → Bar Exam → Pass/Fail
```

On prédira **`y_pass_bar`** : a-t-on réussi le barreau (0/1) ? À partir des seules variables connues **à l'entrée** en faculté de droit (notes pré-droit, statut temps plein, revenu familial, tier de la faculté).

Le dataset est livré déjà nettoyé et déjà splitté (train/test) dans `data/law_school/`. Voir `data/law_school/codebook.md` pour le détail des colonnes.

## Setup

In [ ]:
# !pip install -q scikit-learn pandas matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Chargement des CSV pré-traités (train + test pour la classification)
URL = "https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/data/law_school"

df_train = pd.read_csv(f"{URL}/law_school_for_classification_train.csv")
df_test = pd.read_csv(f"{URL}/law_school_for_classification_test.csv")

print(f"Train : {len(df_train):,} lignes")
print(f"Test  : {len(df_test):,} lignes")
df_train.head()

## 2.1 La notion de feature

Une **feature** (variable explicative, prédicteur…) est une colonne dont le modèle se sert pour faire sa prédiction.

Dans `data/law_school/`, les colonnes suivent une convention :

| Préfixe | Rôle | Colonnes ici |
| :-: | - | - |
| `x_` | features prédictives | `x_lsat`, `x_ugpa`, `x_fulltime`, `x_fam_inc`, `x_tier` |
| `y_` | cible à prédire | `y_pass_bar` |
| `z_` | attribut sensible | `z_white` |

Les `x_*` entrent dans `X`. Le `y_*` est ce qu'on essaye de prédire. Le `z_*` est l'attribut sensible que l'on **isole** : on évite de l'utiliser comme feature directe, et on s'en servira plus tard pour vérifier que le modèle n'est pas injuste (cours de jeudi). Pour ce module, on l'écarte simplement de `X`.

Les modèles **adorent** le numérique, **tolèrent mal** le catégoriel brut. Ici tout est déjà numérique (les catégories ont été encodées en codes ordinaux dans le dataset source), donc on n'a pas à se battre avec du `one-hot encoding`. Veinard·es.

In [ ]:
# Aperçu des types
df_train.dtypes

In [ ]:
# Distribution de la cible
df_train["y_pass_bar"].value_counts(normalize=True).round(3)

### Hack Time

Dans la cellule ci-dessous :
- Affichez `df_train.describe()` pour voir les distributions des features numériques
- Repérez la feature qui a la plus grande dispersion (écart-type / range)
- Comparez la proportion de `z_white == 1` dans le train et dans le test. Est-elle bien préservée par le split ?
- Réfléchissez : dans **votre** domaine de recherche, quelles features auraient un rôle analogue à `x_lsat` et `x_ugpa` (signaux disponibles **avant** l'événement à prédire) ?

In [ ]:
# Votre code ici

## 2.2 Nettoyage et préparation

Règle d'or : **garbage in, garbage out**. Un modèle entraîné sur des données pourries fait des prédictions pourries (et a souvent l'air confiant en le faisant).

Trois grands chantiers, en général :
- Les **valeurs manquantes**
- Les **outliers** (vraies erreurs ou vrais individus extrêmes ?)
- La **normalisation** des numériques (certains modèles y sont très sensibles, d'autres s'en fichent)

Bonne nouvelle : le script `data/law_school/fetch_law_school.py` a déjà fait le sale boulot avant qu'on arrive (drop des NA, recodage, retrait des variables qui causeraient du **target leakage** — par exemple les notes obtenues *pendant* les études de droit, qui rendraient la prédiction triviale et inutile). C'est documenté dans le codebook.

On vérifie quand même, parce que faire confiance aveuglément aux données qu'on nous tend est statistiquement déconseillé.

In [ ]:
# Sanity check : reste-t-il des NA ?
df_train.isna().sum()

### Hack Time

- Avec `df_train.describe()`, regardez les valeurs `min` et `max` de `x_lsat` et `x_ugpa`. Y a-t-il des valeurs aberrantes (un GPA undergrad à 12 sur 4, par exemple) ?
- Tracez l'histogramme de `x_lsat` (avec `df_train["x_lsat"].hist()`). La distribution a-t-elle l'air gaussienne ? Bi-modale ?
- Question conceptuelle : pourquoi est-ce qu'on a **exclu** la note de première année (`zfygpa`) du dataset livré, alors qu'elle prédirait excellemment bien le passage du barreau ?

In [ ]:
# Votre code ici

## 2.3 Train / test split

L'objectif d'un modèle ML n'est **pas** de bien marcher sur les données d'entraînement. C'est de **généraliser** à des données non vues.

D'où la nécessité de **séparer** les données en deux : on entraîne le modèle sur 75 % (le « train ») et on mesure sa qualité sur les 25 % restants (le « test »). Le modèle n'a jamais vu ces 25 % pendant l'entraînement, donc s'il s'en sort bien là-dessus, on peut espérer qu'il marchera aussi sur de vraies nouvelles données.

Ici, le split est **déjà fait** dans les fichiers. C'est une pratique courante : la personne qui prépare les données fournit directement `_train.csv` et `_test.csv`, ce qui garantit que personne ne triche en regardant le test pendant l'exploration.

> Quand on doit le faire soi-même, l'outil standard est `sklearn.model_selection.train_test_split(X, y, test_size=0.25, stratify=y)`. La **stratification** sur la cible garantit que le test set a à peu près la même proportion de réussite au barreau que le train set.

In [ ]:
# Préparer X et y depuis les CSV pré-splittés
FEATURES = ["x_lsat", "x_ugpa", "x_fulltime", "x_fam_inc", "x_tier"]
TARGET = "y_pass_bar"

X_train = df_train[FEATURES]
y_train = df_train[TARGET]
X_test = df_test[FEATURES]
y_test = df_test[TARGET]

print(f"X_train : {X_train.shape}, y_train : {y_train.shape}")
print(f"X_test  : {X_test.shape}, y_test  : {y_test.shape}")
print(f"Taux de réussite train : {y_train.mean():.3f}")
print(f"Taux de réussite test  : {y_test.mean():.3f}")

## 2.4 Overfitting

Le **piège classique du ML** : le modèle apprend ses données par cœur, performe à 100 % sur le train, et plante royalement sur le test.

Symptôme : précision sur le train >> précision sur le test.

L'**arbre de décision** est la machine à fabriquer de l'overfitting : laissé sans contrainte sur sa profondeur, il mémorise littéralement les données d'entraînement.

(Un arbre de décision, c'est une suite de questions en cascade : « LSAT > 35 ? Oui → UGPA > 3.5 ? Non → temps plein ? »… Plus l'arbre est profond, plus il pose de questions, plus il mémorise.)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Squelette : entraîner des arbres de profondeur croissante et observer la divergence
# depths = [1, 3, 5, 10, 20, None]
# train_scores, test_scores = [], []
# for d in depths:
#     clf = DecisionTreeClassifier(max_depth=d, random_state=42)
#     clf.fit(X_train, y_train)
#     train_scores.append(clf.score(X_train, y_train))
#     test_scores.append(clf.score(X_test, y_test))
#
# xs = [d if d is not None else 30 for d in depths]
# plt.plot(xs, train_scores, "o-", label="train")
# plt.plot(xs, test_scores, "o-", label="test")
# plt.xlabel("Profondeur de l'arbre"); plt.ylabel("Précision"); plt.legend()

### Hack Time

- À partir de quelle profondeur l'overfitting commence-t-il à se voir ?
- Avec `max_depth=None`, l'arbre est-il plus performant qu'avec une profondeur modérée ?
- Quelle profondeur maximise la **performance en test** ? (C'est ce qu'on appelle le *sweet spot*.)
- Comme `y_pass_bar` est très déséquilibré (~90 % de réussite), un modèle qui prédit toujours « 1 » obtient déjà ~90 % d'accuracy. Comparez à vos arbres : font-ils mieux ? Si non, c'est un signal qu'il faut regarder d'autres métriques (F1, rappel sur la classe minoritaire, matrice de confusion).

## 2.5 Biais-variance : le seul vrai mystère

Tout modèle est tiraillé entre deux écueils :

- **Biais élevé** : modèle trop simple, il rate les vraies tendances (sous-apprentissage)
- **Variance élevée** : modèle trop sensible aux données particulières du train (sur-apprentissage)

Visuellement, c'est la fameuse courbe en U : trop simple → erreur élevée ; trop complexe → erreur élevée. Le bon modèle est au creux.

| Modèle | Biais | Variance |
|---|---|---|
| Régression logistique | Élevé | Faible |
| Arbre profond | Faible | Élevé |
| Random Forest (forêt d'arbres) | ↘ | ↘ |
| LLM pré-entraîné | Très faible | Énorme, compensée par un pré-entraînement massif |

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Random Forest = une forêt de plein d'arbres de décision dont on moyenne les prédictions.
# C'est la recette empirique qui marche le mieux sur ce type de données tabulaires.

# Squelette pour comparer 3 modèles
# models = {
#     "LogReg (biais ↑, variance ↓)": LogisticRegression(max_iter=1000),
#     "Arbre profond (biais ↓, variance ↑)": DecisionTreeClassifier(max_depth=None, random_state=42),
#     "Random Forest (compromis)": RandomForestClassifier(n_estimators=200, random_state=42),
# }
# for name, m in models.items():
#     m.fit(X_train, y_train)
#     print(f"{name:50s}  train={m.score(X_train, y_train):.3f}  test={m.score(X_test, y_test):.3f}")

### Hack Time

- Quel modèle a le plus gros écart train/test ? C'est votre vainqueur de la variance.
- Quel modèle a la pire performance globale ? C'est probablement votre vainqueur du biais.
- Random Forest gagne presque toujours en pratique. Pourquoi ? (Indice : il fait *plein* d'arbres profonds, et moyenne. C'est de la variance qu'on tue par agrégation.)
- Bonus fairness : pour chaque modèle, calculez le taux de prédiction positive séparément pour `z_white == 1` et `z_white == 0`. Le modèle prédit-il « réussite » à la même fréquence dans les deux groupes ? On y reviendra jeudi.

## Récap module 2

Ces 5 concepts (features · nettoyage · split · overfitting · biais-variance) structurent **tout** le machine learning.
Y compris ce qui se passe dans les LLMs : ils ont une variance vertigineuse, compensée par un pré-entraînement vertigineux.

→ Direction le **module 3** : on retourne au texte pour comprendre comment il devient des **tokens**, le carburant des LLMs.